In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv(r"C:\Users\SHREYAS ROY\Desktop\Manipal Work Documents\yeast.csv")
train = df.sample(frac=0.8, random_state=42)
test  = df.drop(train.index)
# NOTE: No empty cells found in the dataset
test = test.reindex(columns=train.columns, fill_value=0) 
print(train.head())

       mcg   gvh   alm   mit  erl  pox   vac   nuc name
123   0.59  0.60  0.49  0.43  0.5  0.0  0.53  0.31  MIT
432   0.45  0.56  0.51  0.64  0.5  0.0  0.54  0.22  MIT
1033  0.57  0.50  0.40  0.08  0.5  0.0  0.47  0.22  ME3
529   0.53  0.44  0.47  0.22  0.5  0.0  0.53  0.22  NUC
1417  0.38  0.37  0.36  0.14  0.5  0.0  0.54  0.22  ME3


In [3]:
targetCol = 'name'
features = train.drop(columns = [targetCol]).to_numpy()
target = train[targetCol].to_numpy() 
testFeatures = test.drop(columns = [targetCol]).to_numpy()
testCases = test[targetCol].to_numpy()

features = features.astype(np.float64)
testFeatures = testFeatures.astype(np.float64)

In [13]:
def computeEntropy(y):
    if len(y) == 0:
        return 0.0
    _, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    
    probabilities = probabilities[probabilities > 0]
    
    entropy = -np.sum(probabilities * np.log2(probabilities))
    return entropy

def splitDataset(X, nodeIndices, feature, threshold):
    if len(nodeIndices) == 0:
        return np.array([], dtype=int), np.array([], dtype=int)
    
    nodeX = X[nodeIndices]
    mask = nodeX[:, feature] <= threshold
    
    leftIndices = nodeIndices[mask]
    rightIndices = nodeIndices[~mask]
    return leftIndices, rightIndices

def computeInformationGain(X, y, nodeIndices, feature, threshold): # Cost Function  
    leftIndices, rightIndices = splitDataset(X, nodeIndices, feature, threshold)
    
    if len(leftIndices) == 0 or len(rightIndices) == 0:
        return 0.0
    
    yNode = y[nodeIndices]
    yLeft = y[leftIndices]
    yRight = y[rightIndices]
    
    h_parent = computeEntropy(yNode)
    h_left = computeEntropy(yLeft)
    h_right = computeEntropy(yRight)
    
    wLeft = len(yLeft) / len(yNode)
    wRight = 1 - wLeft
    
    informationGain = h_parent - (wLeft * h_left + wRight * h_right)
    return informationGain

def getBestSplit(X, y, nodeIndices):   
    numFeatures = X.shape[1]
    bestFeature = -1
    bestThreshold = None
    bestInformationGain = 0.0
    
    if len(nodeIndices) <= 1:
        return bestFeature, bestThreshold
    
    X_node = X[nodeIndices]
    
    for feature in range(numFeatures):
        featureValues = X_node[:, feature]
        thresholds = np.unique(featureValues) # gives sorted unique values
        if len(thresholds) <= 1: # cannot be separated further
            continue
        candidateThresholds = (thresholds[:-1] + thresholds[1:]) / 2.0 # midpoint b/w adjacent sorted unique values (meaningful partition boundary)
        for threshold in candidateThresholds:
            infoGain = computeInformationGain(X, y, nodeIndices, feature, threshold)
            if infoGain > bestInformationGain:
                bestInformationGain = infoGain
                bestFeature = feature
                bestThreshold = threshold
                
    return bestFeature, bestThreshold

def buildTreeRecursive(X, y, nodeIndices, maxDepth, currentDepth=0):
    uniqueClasses, counts = np.unique(y[nodeIndices], return_counts=True)
    majorityClass = uniqueClasses[np.argmax(counts)] if len(counts) > 0 else None
    
    # Base Case:
    if currentDepth == maxDepth or len(uniqueClasses) <= 1 or len(nodeIndices) == 0:
        return {"isLeaf": True, "class": majorityClass, "samples": len(nodeIndices)}
   
    bestFeature, bestThreshold = getBestSplit(X, y, nodeIndices)
    if bestFeature == -1:
        return {"isLeaf": True, "class": majorityClass, "samples": len(nodeIndices)}

    leftIndices, rightIndices = splitDataset(X, nodeIndices, bestFeature, bestThreshold)
    
    leftChild = buildTreeRecursive(X, y, leftIndices, maxDepth, currentDepth + 1)
    rightChild = buildTreeRecursive(X, y, rightIndices, maxDepth, currentDepth + 1)
    
    return {
        "isLeaf": False,
        "feature": bestFeature,
        "threshold": bestThreshold,
        "left": leftChild,
        "right": rightChild
    }

In [27]:
rootIndices = np.arange(target.shape[0])
tree = buildTreeRecursive(features, target, rootIndices, maxDepth = 8, currentDepth = 0)
print(tree)

{'isLeaf': False, 'feature': 2, 'threshold': np.float64(0.435), 'left': {'isLeaf': False, 'feature': 0, 'threshold': np.float64(0.665), 'left': {'isLeaf': False, 'feature': 2, 'threshold': np.float64(0.405), 'left': {'isLeaf': False, 'feature': 0, 'threshold': np.float64(0.555), 'left': {'isLeaf': False, 'feature': 7, 'threshold': np.float64(0.305), 'left': {'isLeaf': False, 'feature': 3, 'threshold': np.float64(0.38), 'left': {'isLeaf': False, 'feature': 1, 'threshold': np.float64(0.675), 'left': {'isLeaf': False, 'feature': 0, 'threshold': np.float64(0.24), 'left': {'isLeaf': True, 'class': 'MIT', 'samples': 1}, 'right': {'isLeaf': True, 'class': 'ME3', 'samples': 82}}, 'right': {'isLeaf': False, 'feature': 0, 'threshold': np.float64(0.455), 'left': {'isLeaf': True, 'class': 'ME2', 'samples': 2}, 'right': {'isLeaf': True, 'class': 'ME3', 'samples': 2}}}, 'right': {'isLeaf': False, 'feature': 2, 'threshold': np.float64(0.36), 'left': {'isLeaf': True, 'class': 'ME3', 'samples': 1}, 'ri

In [28]:
def predictRow(node, sample):
    if node["isLeaf"]:
        return node["class"]
    if sample[node["feature"]] <= node["threshold"]:
        return predictRow(node["left"], sample)
    else:
        return predictRow(node["right"], sample)

def predictTree(tree, X):
    return np.array([predictRow(tree, sample) for sample in X])

In [29]:
def computeConfusionMatrix(yActual, yPred, classes):
    numClasses = len(classes)
    class_to_idx = {c: i for i, c in enumerate(classes)}
    cm = np.zeros((numClasses, numClasses), dtype=np.int64)
    
    for yt, yp in zip(yActual, yPred):
        if yt in class_to_idx and yp in class_to_idx:
            cm[class_to_idx[yt], class_to_idx[yp]] += 1
            
    return cm

def evaluateMulticlassMetrics(yActual, yPred):
    classes = np.unique(np.concatenate([yActual, yPred]))
    cm = computeConfusionMatrix(yActual, yPred, classes)
    
    totalSamples = np.sum(cm)
    correctPredictions = np.sum(np.diagonal(cm))
    accuracy = correctPredictions / totalSamples if totalSamples > 0 else 0.0

    tp = np.diagonal(cm)
    actualTotals = np.sum(cm, axis=1) # True Positives + False Negatives
    predTotals = np.sum(cm, axis=0)   # True Positives + False Positives
    
    recallPerClass = np.where(actualTotals > 0, tp / actualTotals, 0.0)
    macroRecall = np.mean(recallPerClass)
 
    precisionPerClass = np.where(predTotals > 0, tp / predTotals, 0.0)

    denom_f1 = precisionPerClass + recallPerClass
    f1_per_class = np.where(denom_f1 > 0, 2 * (precisionPerClass * recallPerClass) / denom_f1, 0.0)
    macro_f1 = np.mean(f1_per_class)
    
    # Multi-class Matthews Correlation Coefficient (MCC)
    tSum = actualTotals
    pSum = predTotals
    cov_y_y = totalSamples**2 - np.sum(pSum**2)
    cov_x_x = totalSamples**2 - np.sum(tSum**2)
    
    numerator = totalSamples * correctPredictions - np.dot(tSum, pSum)
    denominator = np.sqrt(cov_x_x * cov_y_y) if (cov_x_x * cov_y_y) > 0 else 1.0
    mcc = numerator / denominator if denominator > 0 else 0.0
    
    return {
        "Accuracy": accuracy,
        "Macro Recall": macroRecall,
        "Macro F1": macro_f1,
        "MCC": mcc,
        "Confusion Matrix": cm,
        "Classes": classes
    }

def kFoldCrossValidation(X, y, k=5, maxDepth=3):
    indices = np.arange(len(y))
    np.random.seed(42)
    np.random.shuffle(indices)
    
    folds = np.array_split(indices, k)
    foldMetrics = []
    
    for i in range(k):
        valIndices = folds[i]
        trainIndices = np.hstack([folds[j] for j in range(k) if j != i])
        
        X_train, y_train = X[trainIndices], y[trainIndices]
        X_val, y_val = X[valIndices], y[valIndices]
        
        rootIndices = np.arange(len(y_train))
        tree = buildTreeRecursive(X_train, y_train, rootIndices, maxDepth=maxDepth, currentDepth=0)
        
        predictions = predictTree(tree, X_val)
        
        metrics = evaluateMulticlassMetrics(y_val, predictions)
        foldMetrics.append(metrics)
        print(f"Fold {i+1} -> Accuracy: {metrics['Accuracy']:.4f}, Macro F1: {metrics['Macro F1']:.4f}, MCC: {metrics['MCC']:.4f}")
        
    print(f"Completed {k}-Fold Cross-Validation successfully.")
    return foldMetrics

In [30]:
_ = kFoldCrossValidation(features, target, k=5, maxDepth=8) # Validated 'train'

C:\Users\SHREYAS ROY\AppData\Local\Temp\ipykernel_9628\1435922326.py:30: RuntimeWarning: invalid value encountered in divide
  f1_per_class = np.where(denom_f1 > 0, 2 * (precisionPerClass * recallPerClass) / denom_f1, 0.0)


Fold 1 -> Accuracy: 0.6008, Macro F1: 0.5198, MCC: 0.4815
Fold 2 -> Accuracy: 0.5294, Macro F1: 0.4631, MCC: 0.4009
Fold 3 -> Accuracy: 0.5612, Macro F1: 0.3843, MCC: 0.4312
Fold 4 -> Accuracy: 0.5443, Macro F1: 0.4100, MCC: 0.3984
Fold 5 -> Accuracy: 0.5865, Macro F1: 0.4588, MCC: 0.4595
Completed 5-Fold Cross-Validation successfully.


C:\Users\SHREYAS ROY\AppData\Local\Temp\ipykernel_9628\1435922326.py:27: RuntimeWarning: invalid value encountered in divide
  precisionPerClass = np.where(predTotals > 0, tp / predTotals, 0.0)


In [31]:
# Test Set Evaluation Metrics
predictions = predictTree(tree, testFeatures)
metrics = evaluateMulticlassMetrics(testCases, predictions)
print(f"Test Set Eval. Metrics -> Accuracy: {metrics['Accuracy']:.4f}, Macro F1: {metrics['Macro F1']:.4f}, MCC: {metrics['MCC']:.4f}")

Test Set Eval. Metrics -> Accuracy: 0.5320, Macro F1: 0.3938, MCC: 0.3938


C:\Users\SHREYAS ROY\AppData\Local\Temp\ipykernel_9628\1435922326.py:27: RuntimeWarning: invalid value encountered in divide
  precisionPerClass = np.where(predTotals > 0, tp / predTotals, 0.0)
C:\Users\SHREYAS ROY\AppData\Local\Temp\ipykernel_9628\1435922326.py:30: RuntimeWarning: invalid value encountered in divide
  f1_per_class = np.where(denom_f1 > 0, 2 * (precisionPerClass * recallPerClass) / denom_f1, 0.0)
